In [2]:
# diagnosisカラムにどんなAF表記があるか確認
pd.read_sql("""
    SELECT DISTINCT diagnosis
    FROM first_diag
    WHERE diagnosis LIKE '%心房細動%'
    LIMIT 20
""", con)

,diagnosis
0,持続性心房細動
1,徐脈性心房細動（平成17年僧帽弁形成術
2,発作性心房細動
3,"H27.5.1), 発作性心房細動, 徐脈頻脈症候"
4,慢性心房細動
5,頻脈性心房細動
6,頻脈性持続性心房細動
7,徐脈性心房細動
8,発作性心房細動心
9,心房細動


In [4]:
# サニティチェック
checks = {
    "①first_diag全患者数": "SELECT COUNT(DISTINCT Patient_ID) FROM first_diag",
    "②HFなし母集団": "SELECT COUNT(*) FROM (SELECT Patient_ID FROM first_diag GROUP BY Patient_ID HAVING MAX(heart_failure)=0)",
    "③last_one_month_diag全患者数": "SELECT COUNT(DISTINCT Patient_ID) FROM last_one_month_diag",
    "④②とlast_one_month_diagの一致数": """
        SELECT COUNT(DISTINCT l.Patient_ID)
        FROM last_one_month_diag l
        WHERE l.Patient_ID IN (
            SELECT Patient_ID FROM first_diag GROUP BY Patient_ID HAVING MAX(heart_failure)=0
        )
    """
}

for label, q in checks.items():
    n = pd.read_sql(q, con).iloc[0,0]
    print(f"{label}: {n}")

①first_diag全患者数: 3708
②HFなし母集団: 2908
③last_one_month_diag全患者数: 5805
④②とlast_one_month_diagの一致数: 2731


In [5]:
pd.read_sql("""
    SELECT hf_developed, COUNT(*) as n
    FROM (
        SELECT 
            af_base.af_flg,
            COALESCE(hf_last.hf_developed, 0) AS hf_developed
        FROM (
            SELECT Patient_ID,
                MAX(CASE WHEN diagnosis LIKE '%心房細動%' THEN 1 ELSE 0 END) AS af_flg
            FROM first_diag
            WHERE Patient_ID IN (
                SELECT Patient_ID FROM first_diag GROUP BY Patient_ID HAVING MAX(heart_failure)=0
            )
            GROUP BY Patient_ID
        ) af_base
        LEFT JOIN (
            SELECT Patient_ID, MAX(heart_failure_flg) AS hf_developed
            FROM last_one_month_diag
            GROUP BY Patient_ID
        ) hf_last USING (Patient_ID)
    )
    GROUP BY hf_developed
""", con)

,hf_developed,n
0,0,2723
1,1,185


In [15]:
pd.read_sql("""
WITH 
base AS (
    SELECT Patient_ID
    FROM first_diag
    GROUP BY Patient_ID
    HAVING MAX(heart_failure) = 0
),
af_base AS (
    SELECT 
        Patient_ID,
        MAX(CASE WHEN COALESCE(diagnosis,'') LIKE '%心房細動%' THEN 1 ELSE 0 END) AS af_flg
    FROM first_diag
    WHERE Patient_ID IN (SELECT Patient_ID FROM base)
    GROUP BY Patient_ID
)
SELECT COUNT(*) as total, COUNT(DISTINCT Patient_ID) as unique_id
FROM af_base
""", con)

,total,unique_id
0,2908,2908


In [16]:
pd.read_sql("""
WITH 
base AS (
    SELECT Patient_ID
    FROM first_diag
    GROUP BY Patient_ID
    HAVING MAX(heart_failure) = 0
),
hf_last AS (
    SELECT 
        Patient_ID,
        MAX(COALESCE(heart_failure_flg, 0)) AS hf_developed
    FROM last_one_month_diag
    WHERE Patient_ID IN (SELECT Patient_ID FROM base)
    GROUP BY Patient_ID
)
SELECT COUNT(*) as total, COUNT(DISTINCT Patient_ID) as unique_id
FROM hf_last
""", con)

,total,unique_id
0,2731,2731


In [19]:
pd.read_sql("""
WITH 
base AS (
    SELECT Patient_ID
    FROM first_diag
    GROUP BY Patient_ID
    HAVING MAX(heart_failure) = 0
),
af_base AS (
    SELECT 
        Patient_ID,
        MAX(CASE WHEN COALESCE(diagnosis,'') LIKE '%心房細動%' THEN 1 ELSE 0 END) AS af_flg
    FROM first_diag
    WHERE Patient_ID IN (SELECT Patient_ID FROM base)
    GROUP BY Patient_ID
),
hf_last AS (
    SELECT 
        Patient_ID,
        MAX(COALESCE(heart_failure_flg, 0)) AS hf_developed
    FROM last_one_month_diag
    WHERE Patient_ID IN (SELECT Patient_ID FROM base)
    GROUP BY Patient_ID
),
joined AS (
    SELECT
        af_base.af_flg,
        COALESCE(hf_last.hf_developed, 0) AS hf_developed
    FROM af_base
    LEFT JOIN hf_last USING (Patient_ID)
)
SELECT af_flg, hf_developed, COUNT(*) as n
FROM joined
GROUP BY af_flg, hf_developed
ORDER BY af_flg, hf_developed
""", con)

,af_flg,hf_developed,n
0,0,0,2540
1,0,1,148
2,1,0,183
3,1,1,37


In [20]:
pd.read_sql("""
SELECT 
    'first_diag' AS source,
    COUNT(DISTINCT Patient_ID) AS hf_patients
FROM first_diag
WHERE Patient_ID IN (
    SELECT Patient_ID FROM first_diag GROUP BY Patient_ID HAVING MAX(heart_failure) = 1
)

UNION ALL

SELECT 
    'last_one_month_diag' AS source,
    COUNT(DISTINCT Patient_ID) AS hf_patients
FROM last_one_month_diag
WHERE Patient_ID IN (
    SELECT Patient_ID FROM last_one_month_diag GROUP BY Patient_ID HAVING MAX(heart_failure_flg) = 1
)
""", con)

,source,hf_patients
0,first_diag,800
1,last_one_month_diag,1175


In [21]:
pd.read_sql("""
SELECT COUNT(DISTINCT l.Patient_ID) as n
FROM last_one_month_diag l
WHERE l.Patient_ID IN (
    SELECT Patient_ID FROM last_one_month_diag
    GROUP BY Patient_ID HAVING MAX(heart_failure_flg) = 1
)
AND l.Patient_ID IN (
    SELECT Patient_ID FROM first_diag
    GROUP BY Patient_ID HAVING MAX(heart_failure) = 0
)
""", con)

,n
0,185


In [22]:
# 190人のPatient_IDを抽出
pd.read_sql("""
SELECT DISTINCT l.Patient_ID
FROM last_one_month_diag l
WHERE l.Patient_ID IN (
    SELECT Patient_ID FROM last_one_month_diag
    GROUP BY Patient_ID HAVING MAX(heart_failure_flg) = 1
)
AND l.Patient_ID NOT IN (
    SELECT DISTINCT Patient_ID FROM first_diag
)
""", con)

,Patient_ID
0,201172
1,191182
2,191685
3,211364
4,191096
...,...
252,222763
253,201080
254,241881
255,191107


In [23]:
# 1. 正確な人数の再確認
pd.read_sql("""
SELECT 
    COUNT(DISTINCT l.Patient_ID) as first_diagに未登録かつlastでHFあり
FROM last_one_month_diag l
WHERE l.Patient_ID IN (
    SELECT Patient_ID FROM last_one_month_diag
    GROUP BY Patient_ID HAVING MAX(heart_failure_flg) = 1
)
AND l.Patient_ID NOT IN (
    SELECT DISTINCT Patient_ID FROM first_diag
)
""", con)

,first_diagに未登録かつlastでHFあり
0,257


In [24]:
# 2. Patient_Masterに存在するか
pd.read_sql("""
SELECT COUNT(DISTINCT Patient_ID) as n_in_master
FROM Patient_Master
WHERE Patient_ID IN (
    SELECT DISTINCT l.Patient_ID
    FROM last_one_month_diag l
    WHERE l.Patient_ID IN (
        SELECT Patient_ID FROM last_one_month_diag
        GROUP BY Patient_ID HAVING MAX(heart_failure_flg) = 1
    )
    AND l.Patient_ID NOT IN (
        SELECT DISTINCT Patient_ID FROM first_diag
    )
)
""", con)

,n_in_master
0,181


In [25]:
# 3. last_one_month_diagの該当行を確認（どんなデータか）
pd.read_sql("""
SELECT *
FROM last_one_month_diag
WHERE Patient_ID IN (
    SELECT DISTINCT l.Patient_ID
    FROM last_one_month_diag l
    WHERE l.Patient_ID IN (
        SELECT Patient_ID FROM last_one_month_diag
        GROUP BY Patient_ID HAVING MAX(heart_failure_flg) = 1
    )
    AND l.Patient_ID NOT IN (
        SELECT DISTINCT Patient_ID FROM first_diag
    )
)
LIMIT 20
""", con)

,Patient_ID,diagnosis,category,heart_failure_flg
0,201172,慢性心不全、永続性心房細動、高血圧,【心疾患】、分類不能,1
1,201172,糖尿病,【糖尿病および内分泌疾患】,0
2,201172,CKD (G3a),【腎疾患】,0
3,201172,肝機能障害,【消化器疾患】,0
4,201172,間質性肺炎,【肺疾患】,0
5,201172,マントル細胞リンパ腫,分類不能,0
6,201172,不眠症,分類不能,0
7,201172,その他、既往歴：今年3月にIP,分類不能,0
8,201172,63歳 マントルリンパ腫、HT,分類不能,0
9,201172,af、家族歴：CVA (-),分類不能,0


In [26]:
# first_diag全患者数 vs last_one_month_diag全患者数 vs Patient_Master全患者数
pd.read_sql("""
SELECT 
    'Patient_Master' as tbl, COUNT(DISTINCT Patient_ID) as n FROM Patient_Master
UNION ALL
SELECT 'first_diag', COUNT(DISTINCT Patient_ID) FROM first_diag
UNION ALL
SELECT 'last_one_month_diag', COUNT(DISTINCT Patient_ID) FROM last_one_month_diag
""", con)

,tbl,n
0,Patient_Master,5238
1,first_diag,3708
2,last_one_month_diag,5805


In [27]:
# Patient_Masterにいるがfirst_diagに未登録の患者数
pd.read_sql("""
SELECT COUNT(DISTINCT Patient_ID) as n
FROM Patient_Master
WHERE Patient_ID NOT IN (
    SELECT DISTINCT Patient_ID FROM first_diag
)
""", con)

,n
0,1532


In [28]:
# ②の確認
pd.read_sql("""
SELECT COUNT(DISTINCT Patient_ID) as n
FROM last_one_month_diag
WHERE Patient_ID NOT IN (
    SELECT DISTINCT Patient_ID FROM Patient_Master
)
""", con)

,n
0,1186
